# Olist Logistics Agent

Foco exclusivo na análise de prazos de entrega, atrasos e performance logística para entender gargalos da cadeia de distribuição.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

sns.set_theme(style='whitegrid')
warnings.filterwarnings('ignore')

## Conectar o Google Drive

Para rodar no Colab, precisamos montar o Drive para acessar os CSVs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Carregar dados relevantes de logística

In [ ]:
base_path = '/content/drive/MyDrive/1AIAT/Tech_Challange/FASE1/archive/'
orders = pd.read_csv(base_path + 'olist_orders_dataset.csv')
order_items = pd.read_csv(base_path + 'olist_order_items_dataset.csv')
sellers = pd.read_csv(base_path + 'olist_sellers_dataset.csv')
sellers['seller_label'] = 'seller_' + sellers['seller_id'].astype(str)

print('Shapes:')
print('orders', orders.shape)
print('order_items', order_items.shape)
print('sellers', sellers.shape)

## Preparar métricas de entrega

Converter datas e calcular tempos de entrega para análise logística.

In [ ]:
orders['order_approved_at'] = pd.to_datetime(orders['order_approved_at'])
orders['order_delivered_carrier_date'] = pd.to_datetime(orders['order_delivered_carrier_date'])
orders['order_delivered_customer_date'] = pd.to_datetime(orders['order_delivered_customer_date'])
orders['order_estimated_delivery_date'] = pd.to_datetime(orders['order_estimated_delivery_date'])
order_items['shipping_limit_date'] = pd.to_datetime(order_items['shipping_limit_date'])

orders['time_to_carrier'] = (orders['order_delivered_carrier_date'] - orders['order_approved_at']).dt.days
orders['time_to_customer'] = (orders['order_delivered_customer_date'] - orders['order_approved_at']).dt.days
orders['carrier_to_customer'] = (orders['order_delivered_customer_date'] - orders['order_delivered_carrier_date']).dt.days

orders[['time_to_carrier', 'time_to_customer', 'carrier_to_customer']].describe()

### Tempo por etapa logística do seller para o cliente

Neste bloco, calculamos os principais tempos da cadeia logística:
- tempo do seller até o repasse ao parceiro logístico (`order_delivered_carrier_date - order_approved_at`);
- tempo do parceiro logístico até a entrega ao cliente (`order_delivered_customer_date - order_delivered_carrier_date`);
- tempo total desde a aprovação do pedido até a entrega ao cliente;
- diferença entre `order_delivered_carrier_date` e `shipping_limit_date` para medir o cumprimento do prazo do seller;
- diferença entre `order_delivered_customer_date` e `order_estimated_delivery_date` para avaliar a entrega contra a previsão ao cliente.

A seguir, vamos comparar graficamente os valores médios de `seller_handling_days` e `logistics_days` para os sellers mais ativos.

A análise considera apenas pedidos com todas as datas necessárias preenchidas, garantindo que os tempos sejam válidos.

In [ ]:
order_sellers = pd.merge(order_items[['order_id', 'seller_id', 'shipping_limit_date']], sellers[['seller_id', 'seller_label']], on='seller_id', how='left')
order_sellers = order_sellers.drop_duplicates(subset=['order_id', 'seller_id'])
order_with_sellers = pd.merge(orders, order_sellers, on='order_id', how='left')

seller_stage_times = (
    order_with_sellers.dropna(subset=['order_delivered_carrier_date', 'order_delivered_customer_date', 'shipping_limit_date'])
    .assign(
        seller_handling_days=lambda df: (df['order_delivered_carrier_date'] - df['order_approved_at']).dt.days,
        logistics_days=lambda df: (df['order_delivered_customer_date'] - df['order_delivered_carrier_date']).dt.days,
        total_days=lambda df: (df['order_delivered_customer_date'] - df['order_approved_at']).dt.days,
        seller_deadline_gap=lambda df: (df['order_delivered_carrier_date'] - df['shipping_limit_date']).dt.days,
        delivery_delay_vs_estimate=lambda df: (df['order_delivered_customer_date'] - df['order_estimated_delivery_date']).dt.days
    )
    .groupby('seller_label')
    .agg(
        avg_seller_handling_days=('seller_handling_days', 'mean'),
        avg_logistics_days=('logistics_days', 'mean'),
        avg_total_days=('total_days', 'mean'),
        avg_seller_deadline_gap=('seller_deadline_gap', 'mean'),
        avg_delivery_delay_vs_estimate=('delivery_delay_vs_estimate', 'mean'),
        order_count=('order_id', 'nunique')
    )
    .reset_index()
)

# Mostrar as primeiras linhas da tabela de métricas
seller_stage_times.head()

# Gráfico comparativo das principais etapas para os top 20 sellers
ranked_sellers = seller_stage_times.sort_values('order_count', ascending=False).head(20)
plot_data = ranked_sellers.melt(
    id_vars='seller_label',
    value_vars=['avg_seller_handling_days', 'avg_logistics_days'],
    var_name='stage',
    value_name='average_days'
)

plt.figure(figsize=(14, 7))
ax = sns.barplot(data=plot_data, x='seller_label', y='average_days', hue='stage', palette='muted')
ax.set_title('Comparação média por seller: seller handling days vs logistics days')
ax.set_xlabel('Seller')
ax.set_ylabel('Dias médios')
ax.tick_params(axis='x', rotation=45)
ax.legend(title='Métrica', labels=['Média seller handling days', 'Média logistics days'])
plt.tight_layout()
plt.show()